## Exploration

This notebook assumes `preprocess.py`, `train.py`, and `test.py` have already been run.

The cells below are for inspecting a completed run rather than creating a new one.
They let you explore three concrete objects:

1. the saved configuration and artifact paths for the current run;
2. the optimisation trace and held-out completion scores saved by the scripts;
3. image-space trajectories, both from the expert data-collection policy and from the learned planner on one held-out start.


In [ ]:
import importlib
import random
import matplotlib.pyplot as plt
import torch
import src.environment as env
import src.pipeline as pipe

env = importlib.reload(env)
pipe = importlib.reload(pipe)


In [ ]:
CFG = pipe.default_cfg()
RUN = pipe.run_dir(CFG)
{
	"cfg": CFG,
	"run_dir": str(RUN),
}


### Saved Metrics

The next cells read saved artifacts from disk.
They do not rerun preprocessing, training, or full testing.

The loss plot is about optimisation behaviour: whether the fitted objective settled or diverged.
The completion summary is about held-out control behaviour: how much of the board the saved planner completed on unseen starts in the last finished run.


In [ ]:
hist = torch.load(RUN / "history.pt", map_location="cpu")
res = torch.load(RUN / "test_results.pt", map_location="cpu")
{
	"last_history_row": hist[-1],
	"test_results": res,
}


In [ ]:
xs = [row["epoch"] for row in hist]
tr = [row["train_loss"] for row in hist]
va = [row["val_loss"] for row in hist]
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(xs, tr, label="train")
ax.plot(xs, va, label="val")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend()
plt.show()


In [ ]:
xs = torch.tensor(res["scores"])
sd = 0.0
if xs.numel() > 1:
	sd = float(xs.std(unbiased=True).item())
{
	"n": int(xs.numel()),
	"mean_completion": float(xs.mean().item()),
	"sd": sd,
	"min": float(xs.min().item()),
	"max": float(xs.max().item()),
}


### Expert Rollout Preview

The next cell shows frames from the perturbed Hamiltonian policy used during preprocessing.
This is exploration of the image-space supervision source: what kinds of board states the world model and evaluator were trained from.
It is not the learned planner yet.


In [ ]:
frames = pipe.preview(CFG, 12)
fig, axes = plt.subplots(1, len(frames), figsize=(2 * len(frames), 2))
axes = [axes] if len(frames) == 1 else axes
for ax, frame in zip(axes, frames):
	ax.imshow(frame.numpy())
	ax.axis("off")
plt.show()


### Learned Planner On One Held-Out Start

The next two cells load `best.pt` and one saved start from `test.pt`, then roll the learned planner forward for a short horizon.
This is local qualitative inspection of the saved checkpoint on a fixed unseen start.
It is useful for asking what the planner seems to prefer visually, but it is not a replacement for the aggregate `test.py` score.


In [ ]:
data = torch.load(RUN / "test.pt", map_location="cpu")
model, dev = pipe.load_model(CFG)
snake = tuple(tuple(int(x) for x in seg.tolist()) for seg in data["starts_snake"][0])
food = tuple(int(x) for x in data["starts_food"][0].tolist())
rng = random.Random(CFG["seed"] + 123)
sim = env.Simulator(CFG["L"], CFG["U"], random.Random(0))
sim.reset(snake, food)
frames = [sim.display().clone()]
acts = []
vals = []
for _ in range(11):
	if not sim.state.alive:
		break
	act, val, _ = pipe.plan_action(model, sim, CFG, rng, dev)
	acts.append(act)
	vals.append(val)
	sim.step(act)
	frames.append(sim.display().clone())
{
	"start_food": food,
	"actions": acts,
	"scores": [round(x, 4) for x in vals],
}


In [ ]:
fig, axes = plt.subplots(1, len(frames), figsize=(2 * len(frames), 2))
axes = [axes] if len(frames) == 1 else axes
for ax, frame in zip(axes, frames):
	ax.imshow(frame.numpy())
	ax.axis("off")
plt.show()
{
	"alive": sim.state.alive,
	"won": sim.state.won,
	"length": len(sim.state.snake),
}
